# Taxi-v3

### Описание задачи

**Характеристики игры**  
* Эпизодическая среда
* Детерминированная среда
* Дискретная среда
* Среда с полной информацией

**Цель игры**  
Необходимо подобрать пассажира в одном из четырёх районов города и высадить его в точке назначения.

**Окружение**

<img src="https://gymnasium.farama.org/_images/taxi.gif" />

**Пространство действий**

* 0: Ехать на юг (вниз)
* 1: Ехать на север (вверх)
* 2: Ехать на восток (вправо)
* 3: Ехать на запад (влево)
* 4: Посадить пассажира
* 5: Высадить пассажира

**Терминальные состояния**  
Игра заканчивается, когда вы доставили пассажира к месту назначения или эпизод длится более **200** шагов.

**Функция вознаграждения**  
* **-1** вычитается за каждый шаг в среде, если нет иной награды
* **+20** за доставленного пассажира
* **-10** за неправерное выполнение действий "посадить пассажира" и "высадить пассажира"

**Полезные ссылки**
* https://gymnasium.farama.org/environments/toy_text/taxi



***

### Установка зависимостей, импорт библиотек

**Установка зависимостей**

In [ ]:
!pip install gymnasium

**Импорт библиотек**

In [ ]:
import os
import sys
import PIL
import time
import json
import random
import numpy as np
import pandas as pd
import shutil as sh
from collections import namedtuple
from pickle import dumps, loads
from glob import glob
from pytz import timezone
from datetime import datetime
from google.colab import drive
from collections import deque
from base64 import b64encode
from IPython.display import display, clear_output, HTML
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
import gymnasium as gym
from gymnasium.core import Wrapper

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Подключение Google диска

In [ ]:
drive.mount('/content/drive')

### Выбор временной зоны

In [ ]:
TZ = timezone('Europe/Moscow')

### Определение путей и директорий

In [ ]:
env_dir = '/content/drive/MyDrive/Мои курсы и лекции/OTUS/Модуль 3. Advanced Reinforcement Learning/Тема 2. Обучение с использованием модели среды (Часть 2)./Taxi'

### Вспомогательные методы

**Отображение состояния среды, сохранение записи эпизода, просмотр записи эпизода**

In [ ]:
def display_state(state):
    plt.figure(figsize=(8, 6))
    plt.imshow(state)
    plt.axis('off')


def record_episode(eps_frames, records_dir, agent_name, exp_id, eps_num):
    record_path = os.path.join(records_dir, f'{agent_name}_{exp_id}_eps-{eps_num}.mp4')
    eps_frame_dir = 'episode_frames'
    os.mkdir(eps_frame_dir)

    for i, frame  in enumerate(eps_frames):
        PIL.Image.fromarray(frame).save(os.path.join(eps_frame_dir, f'frame-{i+1}.png'))

    os.system(f'ffmpeg -r 2 -i {eps_frame_dir}/frame-%1d.png -vcodec libx264 -b 10M -y "{record_path}"');
    sh.rmtree(eps_frame_dir)


def show_episode_records(records_dir):
    record_paths = glob(os.path.join(records_dir, "*.mp4"))
    html_str = ''
    for i, record_path in enumerate(record_paths):
        mp4 = open(record_path, 'rb').read()
        data = f"data:video/mp4;base64,{b64encode(mp4).decode()}"
        html_str += f'EPISODE # {i+1}<br><video width=500 controls><source src="{data}" type="video/mp4"></video><br><br>'
    return HTML(html_str)


**Создание директорий для логирования результатов, сохрание параметров и метрик экспериментов**

In [ ]:
def create_exp_dirs(env_dir, exp_params):
    dirs = dict()
    dirs['exp'] = os.path.join(env_dir, 'experiments', exp_params["algorithm_name"], exp_params["exp_id"])
    dirs['training'] = os.path.join(dirs['exp'], 'training')
    dirs['evaluation'] = os.path.join(dirs['exp'], 'evaluation')
    os.makedirs(dirs['training'], exist_ok=True)
    os.makedirs(dirs['evaluation'], exist_ok=True)
    return dirs


def save_exp_params(params, exp_dir):
    params_path = os.path.join(exp_dir, 'experiment_params.json')
    with open(params_path, 'w') as f:
        json.dump(params, f)


def save_metrics(metrics, metrics_type, train_eps_dir):
    metrics_path = os.path.join(train_eps_dir, f'{metrics_type}_metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f)

## Обзор окружения

In [ ]:
ACTION_MAP = {
    'Down': 0,
    'Up': 1,
    'Right': 2,
    'Left': 3,
    'Pickup': 4,
    'Dropoff': 5
}

INV_ACTION_MAP = {v: k for k, v in ACTION_MAP.items()}
INV_ACTION_MAP

def create_env():
    return gym.make('Taxi-v3', render_mode='rgb_array')

### Создание окружения, старт нового эпизода, отображение начального состояния

In [ ]:
env = create_env()
state, info = env.reset()
state_rgb = env.render()

print(f'Current state: {state}')
display_state(state_rgb)

In [ ]:
taxi_row, taxi_col, passenger_location, destination = env.decode(state)
taxi_row, taxi_col, passenger_location, destination

### Тестирование окружения

Доступные действия:
* Down
* Up
* Right
* Left
* Pickup
* Dropoff

In [ ]:
action_name = 'Pickup'
action = ACTION_MAP[action_name]

next_state, reward, terminated, truncated, info = env.step(action)
next_state_rgb = env.render()

print(f'Current state: {next_state}\nReward: {reward}\nDone: {terminated, truncated}')
display_state(next_state_rgb)

In [ ]:
env.close()

## Тестирование агента со случайной стратегией

### Случайный агент

In [ ]:
class RandomAgent:

  def __init__(self, env, params):
      self._env = env
      self._params = params

  @property
  def name(self):
      return f"{self._params['algorithm_name']}_agent"


  def start_new_episode(self):
      pass


  def choose_action(self, state, mode='exploitation'):
      return self._env.action_space.sample()

### Методы для запуска эпизодов и оценки агента

In [ ]:
def run_episode(env, agent):
    agent.start_new_episode()
    state, info = env.reset()
    step_count = 0
    total_reward = 0
    frames = list()
    done = False

    while not done:
        action = agent.choose_action(state, mode='exploitation')
        next_state, reward, terminated, truncated, info = env.step(action)
        step_count += 1
        total_reward += reward
        done = terminated or truncated
        state = next_state
        frames.append(env.render())

    win = True if reward > 0 else False
    frames.extend([env.render()] * 3)
    return win, total_reward, step_count, frames


def evaluate_agent(env, agent, exp_params, exp_dirs):
    for eps_num in range(1, exp_params['evaluation']['episode_count'] + 1):
        eps_win, total_reward, eps_step_count, eps_frames = run_episode(env, agent)
        record_episode(eps_frames, exp_dirs['evaluation'], exp_params['algorithm_name'], exp_params['exp_id'], eps_num)

        print(f'EPISODE # {eps_num}')
        if eps_win:
            print('Agent reached the Goal 🏆')
        else:
            print('Agent was defeated ☠️')
        print(f'Total reward: {total_reward}')
        print(f'Number of steps: {eps_step_count}')
        print('-' * 50)

### Параметры эксперимента

In [ ]:
exp_params = {
    'exp_id': f'exp-{datetime.now(TZ).strftime("%d%m-%H%M%S")}',
    'algorithm_name': 'random',
    'seed': 21,
    'evaluation': {
        'episode_count': 3
    }
}

exp_dirs = create_exp_dirs(env_dir, exp_params)
save_exp_params(exp_params, exp_dirs['exp'])

### Оценка агента

In [ ]:
env = create_env()
random_agent = RandomAgent(env, exp_params)
evaluate_agent(env, random_agent, exp_params, exp_dirs)
env.close()

In [ ]:
show_episode_records(exp_dirs['evaluation'])

## Тестирование MCTS-стратегии

Monte Carlo tree search (MCTS) is a heuristic search algorithm, which shows cool results in challenging domains such as Go and chess. The algorithm builds a search tree, iteratively traverses it, and evaluates its nodes using a Monte-Carlo simulation.

In this notebook, we'll implement a MCTS planning and use it to solve some Gym envs.

![image.png](https://i.postimg.cc/6QmwnjPS/image.png)

__How it works?__
We just start with an empty tree and expand it. There are several common procedures.

__1) Selection__
Starting from the root, recursively select the node that corresponds to the tree policy.  

There are several options for tree policies, which we saw earlier as exploration strategies: epsilon-greedy, Thomson sampling, UCB-1. It was shown that in MCTS, UCB-1 achieves a good result. Further, we will consider the one, but you can try to use others.

Following the UCB-1 tree policy, we will choose an action that, on one hand, we expect to have the highest return, and on the other hand, we haven't explored much.

$$
\dot{a} = argmax_{a} \dot{Q}(s, a)
$$

$$
\dot{Q}(s, a) = Q(s, a) + C_p \sqrt{\frac{2 \log {N}}{n_a}}
$$

where:
- $N$ - number of times we have visited state $s$,
- $n_a$ - number of times we have taken action $a$,
- $C_p$ - exploration balance parameter, which is performed between exploration and exploitation.

Using Hoeffding inequality for rewards $R \in [0,1]$ it can be shown that optimal $C_p = 1/\sqrt{2}$. For rewards outside this range, the parameter should be tuned. You can experiment with other values.

__2) Expansion__
After the selection procedure, we can achieve a leaf node or node in which we don't complete actions. In this case, we expand the tree by feasible actions and get new state nodes.

__3) Simulation__
How we can estimate node Q-values? The idea is to estimate action values for a given _rollout policy_ by averaging the return of many simulated trajectories from the current node. Simply, we can play with random or some special policy or use some model that can estimate it.

__4) Backpropagation__
The reward of the last simulation is backed up through the traversed nodes and propagates Q-value estimations, upwards to the root.

$$
Q({\text{parent}}, a) = r + \gamma \cdot Q({\text{child}}, a)
$$



### Добавление возможности создания снапшота состояния среды


**Реализация**

In [ ]:
StepResult = namedtuple(
    "step_result",
    ("snapshot", "state", "reward", "terminated", "truncated", "info")
)

class WithSnapshots(Wrapper):
    """
    Creates a wrapper that supports saving and loading environemnt states.
    Required for planning algorithms.

    This class will have access to the core environment as self.env, e.g.:
    - self.env.reset()           #reset original env
    - self.env.ale.cloneState()  #make snapshot for atari. load with .restoreState()
    - ...

    You can also use reset() and step() directly for convenience.
    - s, info = self.reset()                   # same as self.env.reset()
    - s, r, terminated, truncated, info = self.step(action)  # same as self.env.step(action)

    Note that while you may use self.render(), it will spawn a window that cannot be pickled.
    Thus, you will need to call self.close() before pickling will work again.
    """

    def get_snapshot(self, render=False):
        """
        :returns: environment state that can be loaded with load_snapshot
        Snapshots guarantee same env behaviour each time they are loaded.

        Warning! Snapshots can be arbitrary things (strings, integers, json, tuples)
        Don't count on them being pickle strings when implementing MCTS.

        Developer Note: Make sure the object you return will not be affected by
        anything that happens to the environment after it's saved.
        You shouldn't, for example, return self.env.
        In case of doubt, use pickle.dumps or deepcopy.
        """
        # if self.unwrapped.viewer is not None:
        #     self.unwrapped.viewer.close()
        #     self.unwrapped.viewer = None

        if render:
            self.render()
            self.close()

        return dumps(self.env)


    def load_snapshot(self, snapshot, render=False):
        """
        Loads snapshot as current env state.
        Should not change snapshot inplace (in case of doubt, deepcopy).
        """
        assert not hasattr(self, "_monitor") or hasattr(
            self.env, "_monitor"), "can't backtrack while recording"

        if render:
            self.render()
            self.close()

        self.env = loads(snapshot)


    def step_with_snapshot(self, snapshot, action):
        """
        A convenience function that
        - loads snapshot,
        - commits action via self.step,
        - and takes snapshot again :)

        :returns: next snapshot, next_state, reward, is_done, info

        Basically it returns next snapshot and everything that env.step would have returned.
        """
        self.load_snapshot(snapshot)
        state, reward, terminated, truncated, info = self.env.step(action)
        next_snapshot = self.get_snapshot()

        return StepResult(
            next_snapshot,
            state,
            reward,
            terminated,
            truncated,
            info
        )

**Тестирование**

In [ ]:
def test_snapshotting():
    env = WithSnapshots(create_env())
    env.reset()
    snap = env.get_snapshot()
    action = env.action_space.sample()
    state, *_ = env.step(action)
    print(state)

    env.reset()
    env.load_snapshot(snap)
    alt_state, *_ = env.step(action)
    print(alt_state)

    if np.all(state == alt_state):
        print('\nStates are the same :)')
    else:
        print('\nStates are different :(')

test_snapshotting()

In [ ]:
def test_step_with_snapshots():
    env = WithSnapshots(create_env())
    env.reset()
    snap = env.get_snapshot()

    action = 0
    for _ in range(2):
        state, *_ = env.step(action)
        print(state)

    print('\nrepeat steps with snapshot\n')

    for _ in range(2):
        snap, state, *_ = env.step_with_snapshot(snap, action)
        print(state)

test_step_with_snapshots()

### Описание классов для создания корня дерева и дочерних нод

**Реализация нода при построения дерева в алгоритме MCTS**

In [ ]:
class Node:
    """A tree node for MCTS.

    Each Node corresponds to the result of performing a particular action (self.action)
    in a particular state (self.parent), and is essentially one arm in the multi-armed bandit that
    we model in that state.
    """

    parent = None  # parent Node
    qvalue_sum = 0.  # sum of Q-values from all visits (numerator)
    times_visited = 0  # counter of visits (denominator)

    def __init__(self, parent, action, env):
        """
        Creates and empty node with no children.
        Does so by commiting an action and recording outcome.

        :param parent: parent Node
        :param action: action to commit from parent Node
        """

        self.env = env
        self.parent: Node = parent
        self.action = action
        self.children = set()

        step_result = self.env.step_with_snapshot(parent.snapshot, action)
        self.snapshot, self.state, self.immediate_reward, terminated, truncated, _ = step_result
        self.is_done = terminated or truncated


    def is_leaf(self):
        return len(self.children) == 0


    def is_root(self):
        return self.parent is None


    def get_qvalue_estimate(self):
        return self.qvalue_sum / self.times_visited if self.times_visited != 0 else 0


    def adjust_qvalue_with_ucb_score(self, scale=10, max_value=1e30):
        """
        Computes ucb1 upper bound using current value and visit counts for node and it's parent.

        :param scale: Multiplies upper bound by that. From Hoeffding inequality,
                      assumes reward range to be [0, scale].
        :param max_value: a value that represents infinity (for unvisited nodes).
        """
        if self.times_visited == 0:
            return max_value

        N_tot = self.parent.times_visited
        n_a = self.times_visited
        U = np.sqrt(2 * np.log(N_tot) / n_a)

        return self.get_qvalue_estimate() + scale * U


    def select_best_child(self, greedy):
        best_child, best_score = None, None
        for child_node in self.children:
            score = child_node.get_qvalue_estimate() if greedy else child_node.adjust_qvalue_with_ucb_score()

            if best_child is None or score > best_score:
                best_child = child_node
                best_score = score

        return best_child


    def select_best_leaf(self):
        """
        Picks the leaf with the highest priority to expand.
        Does so by recursively picking nodes with the best UCB-1 score until it reaches a leaf.
        """
        if self.is_leaf():
            return self

        best_child = self.select_best_child(greedy=False)

        return best_child.select_best_leaf()


    def expand(self):
        """
        Expands the current node by creating all possible child nodes.
        Then returns one of those children.
        """
        assert not self.is_done, "can't expand from terminal state"

        n_actions = self.env.action_space.n
        for action in range(n_actions):
            self.children.add(Node(self, action, self.env))

        # If you have implemented any heuristics in select_best_leaf(), they will be used here.
        # Otherwise, this is equivalent to picking some undefined newly created child node.
        return self.select_best_leaf()


    def rollout(self, max_rollout_step_count):
        """
        Play the game from this state to the end (done) or for max_rollout_step_count steps.

        On each step, pick action at random (hint: env.action_space.sample()).

        Compute sum of rewards from the current state until the end of the episode.
        """
        self.env.load_snapshot(self.snapshot)
        state = self.state
        is_done = self.is_done

        rollout_reward = 0
        for _ in range(max_rollout_step_count):
            if is_done:
                break
            action = self.env.action_space.sample()
            state, reward, terminated, truncated, _ = self.env.step(action)
            rollout_reward += reward
            is_done = terminated or truncated

        return rollout_reward


    def propagate(self, child_qvalue):
        """
        Uses child Q-value (sum of rewards) to update parents recursively.
        """
        # 1) compute node q_value
        q_value = self.immediate_reward + child_qvalue
        # 2) update self.qvalue_sum and times_visited
        self.qvalue_sum += q_value
        self.times_visited += 1
        # 3) propagate q_value upwards
        if not self.is_root():
            self.parent.propagate(q_value)


    def safe_delete(self):
        """
        Safe delete to prevent memory leak in some python versions.
        """
        del self.parent
        for child in self.children:
            child.safe_delete()
            del child

**Реализация корня при построения дерева в алгоритме MCTS**

In [ ]:
class Root(Node):

    def __init__(self, snapshot, env):
        """
        Creates special node that acts like tree root.
        :snapshot: snapshot (from env.get_snapshot) to start planning from
        """
        self.env = env
        self.parent = None
        self.action = None
        self.children = set()
        self.snapshot = snapshot
        self.immediate_reward = 0
        self.is_done = False

    @staticmethod
    def from_node(node):
        """
        Initializes node as root.
        """
        root = Root(node.snapshot, node.env)
        copied_fields = ["qvalue_sum", "times_visited", "children", "is_done"]
        for field in copied_fields:
            setattr(root, field, getattr(node, field))
        return root

### Реализация алгоритма планирования с помощью MCTS

In [ ]:
def plan_with_mcts(root, planning_iter_count, max_rollout_step_count):
    """
    Builds tree with monte-carlo tree search for 'planning_iter_count' iterations.
    """
    for _ in range(planning_iter_count):
        node = root.select_best_leaf()
        if node.is_done:
            node.propagate(0)
            break
        else:
            node = node.expand()
            episode_return = node.rollout(max_rollout_step_count)
            node.propagate(episode_return)

### Пошаговый запуск алгоритма MCTS

In [ ]:
env_model = WithSnapshots(create_env())
env_model.reset()
env_model_snapshot = env_model.get_snapshot()
env = loads(env_model_snapshot)

root = Root(env_model_snapshot, env_model)

In [ ]:
state_rgb = env.render()

print(f'Current state: {state}')
display_state(state_rgb)

In [ ]:
plan_with_mcts(root, planning_iter_count=100, max_rollout_step_count=200)

for child in root.children:
    print(f'Action: {INV_ACTION_MAP[child.action]}')
    print(f'Q-value of action: {round(child.get_qvalue_estimate(), 2)}')
    print(f'Sum of Q-values of child nodes: {child.qvalue_sum}')
    print(f'Times visited: {child.times_visited}')
    print()

In [ ]:
best_child = root.select_best_child(greedy=True)
action = best_child.action

print(f'Planned action: {INV_ACTION_MAP[action]}')

In [ ]:
state, reward, terminated, truncated, info = env.step(action)
state_rgb = env.render()

print(f'Current state: {state}\nReward: {reward}\nDone: {terminated, truncated}')
display_state(state_rgb)

In [ ]:
for child in root.children:
    if child != best_child:
        child.safe_delete()

root = Root.from_node(best_child)

In [ ]:
env_model.close()
env.close()

### Запуск алгоритма MCTS для эпизода

In [ ]:
planning_iter_count = 50
eps_max_step_count = 200

env_model = WithSnapshots(create_env())
env_model.reset()
env_model_snapshot = env_model.get_snapshot()
env = loads(env_model_snapshot)

root = Root(env_model_snapshot, env_model)

total_reward = 0
for step_num in range(1, eps_max_step_count+1):
    plan_with_mcts(root, planning_iter_count, eps_max_step_count)
    best_child = root.select_best_child(greedy=True)
    action = best_child.action
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    print(f'Step # {step_num}\tAction: {INV_ACTION_MAP[action]}')
    if terminated:
        print(f'\nAgent reached the Goal 🏆, total reward: {total_reward}')
        break
    if truncated:
        print(f'\nAgent was defeated ☠️, total reward: {total_reward}')
        break

    for child in root.children:
        if child != best_child:
            child.safe_delete()

    root = Root.from_node(best_child)

env_model.close()
env.close()

### MCTS-агент

In [ ]:
class MCTSAgent:

  def __init__(self, params):
      self._params = params

  @property
  def name(self):
      return f"{self._params['algorithm_name']}_agent"


  def start_new_episode(self, env_model):
      env_model_snapshot = env_model.get_snapshot()
      self._root = Root(env_model_snapshot, env_model)
      return loads(env_model_snapshot)

  def choose_action(self, state, mode='exploitation'):
      plan_with_mcts(self._root, self._params['planning_iter_count'], self._params['max_rollout_step_count'])
      best_child = self._root.select_best_child(greedy=True)
      action = best_child.action

      for child in self._root.children:
          if child != best_child:
              child.safe_delete()

      self._root = Root.from_node(best_child)

      return action


In [ ]:
def run_episode(env, agent):
    env_model = WithSnapshots(create_env())
    state, info = env_model.reset()
    step_count = 0
    total_reward = 0
    frames = list()
    done = False
    env = agent.start_new_episode(env_model)

    while not done:
        action = agent.choose_action(state, mode='exploitation')
        next_state, reward, terminated, truncated, info = env.step(action)
        step_count += 1
        total_reward += reward
        done = terminated or truncated
        state = next_state
        frames.append(env.render())

    env_model.close()

    win = True if reward > 0 else False
    frames.extend([env.render()] * 3)
    return win, total_reward, step_count, frames


### Параметры эксперимента

In [ ]:
exp_params = {
    'exp_id': f'exp-{datetime.now(TZ).strftime("%d%m-%H%M%S")}',
    'algorithm_name': 'mcts',
    'seed': 21,
    'planning_iter_count': 50,
    'max_rollout_step_count': 200,
    'evaluation': {
        'episode_count': 3
    }
}

exp_dirs = create_exp_dirs(env_dir, exp_params)
save_exp_params(exp_params, exp_dirs['exp'])

### Оценка агента

In [ ]:
mcts_agent = MCTSAgent(exp_params)
evaluate_agent(None, mcts_agent, exp_params, exp_dirs)